# Week 6 Assignment: Build a Reusable Shipment Prediction Workflow

Extend the Week 5 shipment preparation workflow. Split the labeled shipment rows first. Then place the preprocessing steps and supplied regression estimator in one scikit-learn Pipeline. Fit on the training portion, predict for held-out test shipments, and use the same fitted workflow for separate future shipments with no known target.

This small instructional dataset is for practicing training and workflow mechanics. You will compare actual and predicted values for the held-out rows, but you are not asked to calculate evaluation metrics, compare models, or tune the estimator.


## Workflow specification

- Target: `delivery_hours`
- Identifier to exclude: `shipment_id`
- Nominal features: `origin_region`, `transport_mode`
- Ordinal features: `priority_level`, `service_tier`
- Numerical features: `package_weight_kg`, `distance_miles`, `package_count`
- Priority order: standard < expedited < critical
- Service tier order: economy < regular < premium
- Unknown nominal values: ignore during transformation
- Unknown ordinal values: encode as -1
- Numerical transformation: StandardScaler
- Supplied estimator: LinearRegression
- Split raw X and y before fitting, using `test_size=0.25` and `random_state=42`.
- Fit the Pipeline on `X_train` and `y_train` only.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = Path("data/week6_assignment_shipments.csv")
shipments = pd.read_csv(DATA_PATH)
shipments.head()


## Part 1: Separate features and target, then split

Use the specification above. `X` should contain only the documented predictors. Keep `y` separate. Split these raw rows before you build or fit the preprocessor. The held-out test subset will stand in for shipments the workflow has not learned from.


In [ ]:
target_column = "delivery_hours"
identifier_column = "shipment_id"

# TODO: Complete the feature lists.
nominal_features = []
ordinal_features = []
numeric_features = []

priority_order = ["standard", "expedited", "critical"]
service_order = ["economy", "regular", "premium"]

feature_columns = nominal_features + ordinal_features + numeric_features
assert len(feature_columns) == 7, "Complete the three feature lists first"

# TODO: Create X from feature_columns and y from target_column.
X = shipments[feature_columns]
y = shipments[target_column]

# TODO: Split raw X and y before fitting any preprocessing step.
# Use test_size=0.25 and random_state=42.
X_train, X_test, y_train, y_test = None, None, None, None


## Part 2: Create the column-specific preprocessing step

Route each feature group to the required transformer. The ColumnTransformer definition specifies the operations; the Pipeline will fit data-dependent settings using only training rows. Keep `remainder="drop"` so the identifier and unlisted columns do not enter the transformed data.


In [ ]:
# TODO: Build a ColumnTransformer named preprocessor.
# Use OneHotEncoder(handle_unknown="ignore", sparse_output=False) for nominal features.
# Use OrdinalEncoder with the supplied orders, unknown_value=-1, and
# handle_unknown="use_encoded_value" for ordinal features.
# Use StandardScaler for numerical features.
# Use branch names "nominal", "ordinal", and "numeric".
# Set verbose_feature_names_out=True and request pandas output.
preprocessor = None


## Part 3: Build and fit the complete Pipeline

Put the named preprocessor first and LinearRegression() last. The Pipeline should accept the original feature DataFrame. Fit it using `X_train` and `y_train` only; do not fit any step on `X_test` or `y_test`.


In [ ]:
# TODO: Build workflow using Pipeline with steps named "preprocessor" and "model".
workflow = None

# TODO: Fit the complete workflow using X_train and y_train only.


## Part 4: Predict for the held-out test shipments

Pass the original feature columns in `X_test` to the fitted Pipeline. It applies preprocessing settings learned from the training rows, then predicts. Compare these predictions with `y_test`, which was held back from fitting.


In [ ]:
# TODO: Predict for the held-out X_test rows using the fitted workflow.
test_predictions = None

test_results = X_test.copy()
test_results.insert(0, f"actual_{target_column}", y_test)
test_results[f"predicted_{target_column}"] = test_predictions
test_results


## Part 5: Predict future shipments and verify

The future rows below have no known `delivery_hours` target. Use the same fitted Pipeline to predict for them; do not fit the workflow again. Then verify the split, transformed structures, predictions, and specified unknown-category behavior.


In [ ]:
future_shipments = pd.DataFrame(
    {
        "origin_region": ["north", "central"],
        "transport_mode": ["air", "drone"],
        "priority_level": ["expedited", "urgent"],
        "service_tier": ["premium", "express"],
        "package_weight_kg": [22.0, 150.0],
        "distance_miles": [900, 2500],
        "package_count": [4, 10],
    }
)

# TODO: Predict for the future rows using the fitted workflow.
future_predictions = None
future_results = future_shipments.copy()
future_results[f"predicted_{target_column}"] = future_predictions

fitted_preprocessor = workflow.named_steps["preprocessor"]
feature_names = list(fitted_preprocessor.get_feature_names_out())
X_train_prepared = fitted_preprocessor.transform(X_train)
X_test_prepared = fitted_preprocessor.transform(X_test)
future_prepared = fitted_preprocessor.transform(future_shipments)

# TODO: Verify all of the following:
# - train and test indices are disjoint and together cover the original X rows.
# - shipment_id and delivery_hours do not appear in X.
# - transformed training, test, and future data have the expected rows/columns.
# - test and future prediction counts align with their input rows.
# - all predictions are finite.
# - the unseen ordinal priority_level in future_shipments is represented by -1.
# Finish with print("All Week 6 assignment checks passed.")


## Short response

Answer in your own words:

1. Why do we split the raw data before fitting the preprocessor or model?
2. What does the preprocessor learn from `X_train`, and what does the estimator learn from transformed `X_train` and `y_train`?
3. Why can we compare `test_predictions` with `y_test`, but not evaluate `future_predictions` yet?
4. Explain how `workflow.predict(X_test)` applies the fitted preprocessing.
5. Identify one advantage of keeping preprocessing and the estimator in one Pipeline.
6. What does the unseen ordinal value -1 mean, and what does it not tell you?
7. Look at `future_results`. Are the predicted delivery hours reasonable for each of the two future shipments? Support your answer with evidence from the data, such as the range of `delivery_hours` in `y_train` and how each future row compares to the training rows.
8. A teammate scales `package_weight_kg`, `distance_miles`, and `package_count` on the full dataset first, and then splits it. What goes wrong, and why does the test set no longer stand in for shipments the workflow has never seen?
